In [1]:
import numpy as np
import pandas as pd
from app.data.database import get_engine, canonical_rev_query
from app.core.config import settings

# time series eda

In [52]:
engine = get_engine()
# local_engine = get_engine(True)

selects = f"""
SELECT
    ams.invoice_date,
    amsl.quantity, amsl.price_subtotal, amsl.nepa_cogs_price, amsl.discount, amsl.standard_price, amsl.price_unit,
    CASE WHEN ams.journal_id in ({settings.DB_van_journals}) 
        THEN 'Van_:' || ams.journal_id::text
        ELSE 'Warehouse_:' || wss.warehouse_id::text
    END AS warehouse_label,
    wss.warehouse_id, wss.journal_id, wss.company_id, ams.partner_id, amsl.product_id, ams.move_type
"""
chunks = []
dates = ["2023-12-01", "2024-12-01", "2025-12-01", "2026-09-13"]
for i in range(len(dates)-1):
    query = canonical_rev_query(selects, start_date= dates[i], end_date=dates[i+1])

    print("connecting to nepa db")
    for chunk in pd.read_sql(query, engine, chunksize=500000):
        chunks.append(chunk)
        print(f"Loaded chunk with {len(chunk)} rows")
    
engine.dispose()

df_kpi_forecasting: pd.DataFrame = pd.concat(chunks, ignore_index=True)

connecting to nepa db
Loaded chunk with 500000 rows
Loaded chunk with 500000 rows
Loaded chunk with 38902 rows
connecting to nepa db
Loaded chunk with 500000 rows
Loaded chunk with 500000 rows
Loaded chunk with 500000 rows
Loaded chunk with 500000 rows
Loaded chunk with 373589 rows
connecting to nepa db
Loaded chunk with 500000 rows
Loaded chunk with 500000 rows
Loaded chunk with 500000 rows
Loaded chunk with 441263 rows


In [ ]:
df_kpi_dated = df_kpi_forecasting.copy()

df_kpi_dated["invoice_date"] = pd.to_datetime(
    df_kpi_dated["invoice_date"]
)

# df_kpi_dated.drop(["quantity", "price_subtotal", "journal_id", "company_id", "standard_price"], axis=1, inplace=True)

In [ ]:
df_explore = (
    df_kpi_dated.groupby("invoice_date", as_index=False)
    .agg(
        nepa_cogs_price_sum=("nepa_cogs_price", "sum"),
        price_unit_sum=("price_unit", "sum"),
        price_unit_mean=("price_unit", "mean"),
        revenue_signed_sum=("revenue_signed", "sum"),
        qty_signed_sum=("qty_signed", "sum"),
        discount_sum=("discount", "sum"),
        product_id_unique=("product_id", "nunique"),
        active_customer=("partner_id", "nunique"),
        order_count=("move_type", lambda x: (x == "out_invoice").sum()),
        refund_count=("move_type", lambda x: (x == "out_refund").sum()),
    )
)

# df_explore.columns = [
#     f"{col[0]}_{col[1]}" if col[1] else col[0] 
#     for col in df_explore.columns
# ]

df_explore = df_explore.sort_values(by="invoice_date").reset_index(drop=True)


## exploration and feature testing

In [ ]:
# proper indexing for time series
full_date_range = pd.date_range(
    start=df_explore["invoice_date"].min(),
    end=df_explore["invoice_date"].max(),
    freq="D"
)

df_explore = (
    df_explore.set_index('invoice_date')
    .reindex(full_date_range)
    .ffill()
    .rename_axis("invoice_date")
    .reset_index()
)

In [ ]:
df_explore[df_explore["invoice_date"]=='2025-09-20']

In [ ]:
# date based features and target creation
# creating target
df_explore = df_explore.sort_values(by="invoice_date")

# revenue target
df_explore["target"] = df_explore["revenue_signed_sum"].shift(-1)
df_explore["target_diff"] = (df_explore["target"]-df_explore["revenue_signed_sum"])

# qty target
df_explore["target_qty"] = df_explore["qty_signed_sum"].shift(-1)
df_explore["target_qty_diff"] = (df_explore["target_qty"]-df_explore["qty_signed_sum"])

# cust tgt
df_explore["target_cust"] = df_explore["active_customer"].shift(-1)

# order tgt
df_explore["target_order"] = df_explore["order_count"].shift(-1)

# refund count
df_explore["target_refund"] = df_explore["refund_count"].shift(-1)

# date features
df_explore["month"] = df_explore["invoice_date"].dt.month_name()
df_explore["day_of_week"] = df_explore["invoice_date"].dt.day_name()
# df_explore = df_explore[df_explore["day_of_week"] != "sunday"]

# month features
df_explore["month"] = df_explore["invoice_date"].dt.month
df_explore["month_tgt_enc"] =  df_explore.groupby("month")["target"].transform(lambda x: x.shift(1).expanding(min_periods=1).mean())

# week of day features
df_explore["dow_tgt_enc"] =  df_explore.groupby("day_of_week")["target"].transform(lambda x: x.shift(1).expanding(min_periods=1).mean())
df_explore["day_of_week_sma7W"] = (df_explore.groupby("day_of_week"))["target"].transform(lambda x: x.shift(1).rolling(window=7, min_periods=1).mean())

dow_sma = [2,3,4,5,6,7,8,12,14,52,104]

for N_week in dow_sma:
    df_explore[f"day_of_week_sma{N_week}W"] = (df_explore.groupby("day_of_week"))["target"].transform(lambda x: x.shift(1).rolling(window=N_week, min_periods=1).mean())

# SMA features
cols = ["qty_signed_sum", "revenue_signed_sum", "active_customer", "refund_count", "order_count"]
prefix_map = {
    "nepa_cogs_price_sum" : "cogs",
    "revenue_signed_sum" : "rev",
    "qty_signed_sum" : "qty", 
    "standard_price_sum": "sp_sum",
    "price_unit_sum": "pu_sum"
}
sma_duration = [7, 14, 28, 28*6]
for col in cols:
    prefix = prefix_map.get(col,col)
    for i in sma_duration:
        df_explore[f"{prefix}_{i}d_ma"] = (df_explore[col].shift(1).rolling(window=i, min_periods=1).mean())


# LAG features
from statsmodels.graphics.tsaplots import pacf

lag_duration = {}
for col in cols:
    pacf_vals, confint = pacf(
        df_explore[col],
        nlags=32,
        method="ywm",
        alpha=0.05
    )

    pacf_df = pd.DataFrame({
        "lag": range(len(pacf_vals)),
        "pacf": pacf_vals,
        "lower_ci": confint[:, 0],
        "upper_ci": confint[:, 1]
    })

    pacf_df["significant"] = (
        (pacf_df["lower_ci"] > 0) |
        (pacf_df["upper_ci"] < 0)
    )
    prefix = prefix_map.get(col,col)
    is_valid_lag = pacf_df["significant"] & (pacf_df["lag"] > 5)
    lag_duration[prefix] =  (pacf_df.loc[is_valid_lag, "lag"]+1).tolist() # in relation to the target (t) the lags ness to be moved up due to their data being source starting t-1
    
    
    for i in lag_duration[prefix]:
        df_explore[f"{prefix}_lag_{i}"] = df_explore[col].shift(i-1) 

In [ ]:
df_explore.head()

In [ ]:
w_cols = ['rev_7d_ma', 'rev_14d_ma', 'rev_28d_ma', 'rev_168d_ma', "rev_lag_7", "day_of_week_sma7W",
        "day_of_week_sma8W", "day_of_week_sma12W", "day_of_week_sma14W", "day_of_week_sma52W", "day_of_week_sma104W",
        "day_of_week_sma2W", "day_of_week_sma3W", "day_of_week_sma4W", "day_of_week_sma5W", "day_of_week_sma6W"]
def check_wape():
        sum_actuals = np.sum(np.abs(df_explore["target"]))
        wape = np.sum(np.abs(df_explore["target"] - df_explore[w_col])) / sum_actuals if sum_actuals != 0 else np.nan
        print(f"wape {w_col}: {wape}")
for w_col in w_cols:
    check_wape()



In [ ]:
df_explore[["price_unit_mean","qty_signed_sum"]].corr("pearson")

In [ ]:
import plotly.express as px
import plotly.subplots as sp
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 100)


for col in cols:
    prefix = prefix_map.get(col, col)

    fig = sp.make_subplots(rows=5, cols=1, subplot_titles=[f"{prefix}_{i}d_ma vs target" for i in sma_duration])

    for idx, i in enumerate(sma_duration, start=1):
        ma_col = f"{prefix}_{i}d_ma"
        
        fig.add_trace(
            go.Scatter(x=df_explore["invoice_date"], y=df_explore[ma_col], name= f"{i}d MA"),
            row=idx, col=1
        )
        fig.add_trace(
            go.Scatter(x=df_explore["invoice_date"], y=df_explore["target_"], name="target"),
            row=idx, col=1
        )
        fig.add_trace(
            go.Scatter(x=df_explore["invoice_date"], y=df_explore["target_qty"], name="target_qty"),
            row=idx, col=1
        )
    
    
    fig.update_layout(height=1300, title_text=f"MA of {col}")
    fig.show()

In [ ]:
for col in cols:
    prefix = prefix_map.get(col, col)

    fig = sp.make_subplots(rows=len(lag_duration[prefix]), cols=1, subplot_titles=[f"{prefix}_lag_{i} vs targetz" for i in lag_duration[prefix]])

    for idx, i in enumerate(lag_duration[prefix], start=1):
        lag_col = F"{prefix}_lag_{i}"
        fig.add_trace(
            go.Scatter(x=df_explore["invoice_date"], y=np.log(df_explore[lag_col]), name= f"{i}d MA"),
            row=idx, col=1
        )
        fig.add_trace(
            go.Scatter(x=df_explore["invoice_date"], y=np.log(df_explore["target"]), name="target"),
            row=idx, col=1
        )
        fig.add_trace(
            go.Scatter(x=df_explore["invoice_date"], y=np.log(df_explore["target_qty"]), name="target_qty"),
            row=idx, col=1
        )
    
    
    fig.update_layout(height=3000, title_text=f"lag of {col}")
    fig.show()




In [ ]:
fig1 = sp.make_subplots(rows=1,cols=1)

fig1.add_trace(
    go.Scatter(x=df_explore["invoice_date"], y=df_explore["qty_signed_sum"], name="qty"),
    row=1, col=1
)



# fig1.show()

fig2 = sp.make_subplots(rows=1,cols=1)

fig2.add_trace(
    go.Scatter(x=df_explore["invoice_date"], y=np.log(df_explore["qty_signed_sum"]), name="qty"),
    row=1, col=1
)

# fig2.add_trace(
#     go.Scatter(x=df_explore["invoice_date"], y=np.log(df_explore["revenue_signed_sum"]), name="rev")
# )

# fig2.add_trace(
#     go.Scatter(x=df_explore["invoice_date"], y=np.log(df_explore["active_customer"]), name="cust"),
#     row=1, col=1
# )

# fig2.add_trace(
#     go.Scatter(x=df_explore["invoice_date"], y=np.log(df_explore["order_count"]), name="ord")
# )


# fig2.add_trace(
#     go.Scatter(x=df_explore["invoice_date"], y=np.log(df_explore["refund_count"]), name="ref")
# )

# fig2.add_trace(
#     go.Scatter(x=df_explore["invoice_date"], y=np.log(df_explore["product_id_unique"]), name="productids")
# )

fig2.show()



In [ ]:
def CV(col: pd.DataFrame):
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        print(col.std()/col.mean())

cate = ["day_of_week", "month", "invoice_date"]
CV(df_explore.loc[df_explore['day_of_week'] != 'sunday', [c for c in df_explore.columns if c not in cate]])


# CLUSTERING PRODUCTS

In [2]:
engine = get_engine()
# local_engine = get_engine(True)


chunks = []
dates = ["2023-12-01", "2026-09-13"]
for i in range(len(dates)-1):
    start_date, end_date = dates[i], dates[i+1]
    query = f"""
    WITH daily AS (
        SELECT 
            pt.id AS product_tmpl_id,
            ams.invoice_date,
            SUM(CASE WHEN ams.move_type = 'out_refund'
                THEN -amsl.price_subtotal
                ELSE amsl.price_subtotal
            END) AS revenue_signed,
            SUM(CASE WHEN ams.move_type = 'out_refund'
                THEN -amsl.quantity
                ELSE amsl.quantity
            END) AS qty_signed,
            AVG(amsl.discount) AS discount,
            COUNT(DISTINCT CASE WHEN ams.move_type = 'out_invoice' THEN ams.id END) AS order_count,
            COUNT(DISTINCT CASE WHEN ams.move_type = 'out_refund'  THEN ams.id END) AS refund_count,
            COUNT(DISTINCT ams.partner_id) AS distinct_partners
        FROM account_move_sales_line amsl
        JOIN account_move_sales ams ON ams.id = amsl.move_id
        JOIN product_product pp ON pp.id = amsl.product_id
        JOIN product_template pt ON pt.id = pp.product_tmpl_id
        WHERE ams.state = 'posted'
        AND amsl.account_id IN ({settings.DB_rev_accounts})
        AND pp.type = 'consu'
        AND ams.partner_id NOT IN ({settings.DB_partners_exclude})
        AND ams.invoice_date >= '{start_date}' 
        AND ams.invoice_date < '{end_date}'
        GROUP BY pt.id, ams.invoice_date
    )
    SELECT
        pt.id,
        pt.name AS template_name,
        AVG(d.revenue_signed) AS revenue_signed,
        AVG(d.qty_signed) AS qty_signed,
        AVG(d.discount) AS discount,
        AVG(d.order_count) AS order_count,
        AVG(d.refund_count) AS refund_count,
        AVG(d.distinct_partners) AS active_customers,
        pc.name AS category_name
    FROM daily d
    JOIN product_template pt ON pt.id = d.product_tmpl_id
    JOIN product_category pc ON pc.id = pt.categ_id
    GROUP BY pt.id, pt.name, pc.name
    ORDER BY pt.id ASC;
    """

    print("connecting to nepa db")
    for chunk in pd.read_sql(query, engine, chunksize=500000):
        chunks.append(chunk)
        print(f"Loaded chunk with {len(chunk)} rows")
    
engine.dispose()

df_product_clustering: pd.DataFrame = pd.concat(chunks, ignore_index=True)

connected successfully; version PostgreSQL 17.11 (Ubuntu 17.11-1.pgdg22.04+2) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0, 64-bit
connecting to nepa db
Loaded chunk with 10041 rows


### exploring

In [4]:
df_product_clustering.info()

<class 'pandas.DataFrame'>
RangeIndex: 10189 entries, 0 to 10188
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                10189 non-null  int64  
 1   template_name     10189 non-null  str    
 2   revenue_signed    10189 non-null  float64
 3   qty_signed        10189 non-null  float64
 4   discount          10189 non-null  float64
 5   order_count       10189 non-null  float64
 6   refund_count      10189 non-null  float64
 7   active_customers  10189 non-null  float64
 8   category_name     10189 non-null  str    
dtypes: float64(6), int64(1), str(2)
memory usage: 1.1 MB


In [ ]:
df_product_clustering.isnull().sum()

In [ ]:
df_product_clustering["category_name"].astype("string")
df_product_clustering["template_name"].astype("string")
df_product_clustering.set_index("id", inplace=True)


In [ ]:
df_product_clustering.describe()

In [ ]:
df_product_clustering.nunique()

In [ ]:
df_product_clustering.head()

In [ ]:
df_product_clustering.tail()

In [51]:
df_product_clustering.corr(numeric_only=True, method="spearman")

,id,revenue_signed,qty_signed,discount,order_count,refund_count,active_customers
id,1.000000,0.058121,-0.014060,-0.114341,-0.079861,-0.132989,-0.096911
revenue_signed,0.058121,1.000000,0.278604,0.171034,0.330580,0.158315,0.298715
qty_signed,-0.014060,0.278604,1.000000,0.192956,0.511776,0.168968,0.485119
discount,-0.114341,0.171034,0.192956,1.000000,0.252436,0.246914,0.283438
order_count,-0.079861,0.330580,0.511776,0.252436,1.000000,0.252457,0.914792
refund_count,-0.132989,0.158315,0.168968,0.246914,0.252457,1.000000,0.441930
active_customers,-0.096911,0.298715,0.485119,0.283438,0.914792,0.441930,1.000000


In [ ]:
from plotly import subplots as sp
from plotly import graph_objects as go

fig = sp.make_subplots(
    rows=len(X.columns),
    cols=1,
    subplot_titles=[f"Distribution of {col}" for col in X.columns],
)

for row, col in enumerate(X.columns, start=1):
    fig.add_trace(
        go.Histogram(x=X[col], name=col),
        row=row,
        col=1,
    )

fig.update_layout(
    title_text="Feature distributions",
    height=max(300, 250 * len(X.columns)),
    width=800,
    showlegend=False,
)
fig.show()

## feature engineering/transforming

In [ ]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
df_product_clustering.head()

In [ ]:
include = ["template_name", "category_name", "revenue_signed", "qty_signed", "order_count", "refund_count", "active_customers"]
exclude = ["template_name", "category_name"]



feature_cols = [c for c in df_product_clustering.columns if c in include and c not in exclude]
X = df_product_clustering[feature_cols].copy() 

from sklearn.preprocessing import QuantileTransformer
X = QuantileTransformer(output_distribution='normal').fit_transform(X)

from sklearn.preprocessing import RobustScaler
preprocessor = make_pipeline(
    RobustScaler()
)

X_scaled = preprocessor.fit_transform(X)

clustered = df_product_clustering.copy()

In [9]:
X_scaled.shape

(10189, 5)

## kmeans

In [ ]:
kmeans = KMeans(n_clusters=24, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
clustered["kmeans_labels"] = kmeans_labels

silhoutte_score = silhouette_score(X_scaled, kmeans_labels)
print(f"KMeans silhouette score: {silhoutte_score}")

In [28]:
import numpy as np
from joblib import Parallel, delayed
from sklearn.cluster import KMeans

def balance_score(labels, k):
    counts = np.bincount(labels, minlength=k)
    uniform = len(labels) / k
    deviation = abs(counts.min() - uniform) / uniform
    cv = counts.std() / counts.mean()

    max_min_ratio = counts.max() / max(counts.min(), 1)
    return cv, max_min_ratio, counts.min(), counts.max(), deviation

def fit_kmeans(k, X_scaled):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    cv, max_min_ratio, min_count, max_count, deviation = balance_score(labels, k)
    return k, cv, max_min_ratio, min_count, max_count, deviation, kmeans

results = Parallel(n_jobs=-1, verbose=1)(
    delayed(fit_kmeans)(k, X_scaled) for k in range(13, 301)
)

results_df = pd.DataFrame(results, columns=[
    'k', 'cv', 'max_min_ratio', 'min_cluster', 'max_cluster', 'deviation', 'model'
])

# best = lowest cv (most uniform distribution)
results_df = results_df.sort_values('cv')

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done 248 tasks      | elapsed:   12.4s
[Parallel(n_jobs=-1)]: Done 288 out of 288 | elapsed:   16.0s finished


In [41]:
results_df[['k', 'cv', 'max_min_ratio', 'min_cluster', 'max_cluster', 'deviation', 'model']].sort_values(['cv', 'deviation']).head(20)

,k,cv,max_min_ratio,min_cluster,max_cluster,deviation,model
11,24,0.378495,9.305882,85,791,0.796833,"KMeans(n_clusters=24, n_init=10, random_state=42)"
6,19,0.391551,8.955556,90,806,0.829698,"KMeans(n_clusters=19, n_init=10, random_state=42)"
0,13,0.394889,13.254717,106,1405,0.862763,"KMeans(n_clusters=13, n_init=10, random_state=42)"
9,22,0.425836,24.060606,33,794,0.927696,"KMeans(n_clusters=22, n_init=10, random_state=42)"
4,17,0.427562,11.978022,91,1090,0.845932,"KMeans(n_clusters=17, n_init=10, random_state=42)"
10,23,0.435621,23.264706,34,791,0.922119,"KMeans(n_clusters=23, n_init=10, random_state=42)"
13,26,0.441354,23.235294,34,790,0.911961,"KMeans(n_clusters=26, n_init=10, random_state=42)"
1,14,0.444307,14.505495,91,1320,0.873120,"KMeans(n_clusters=14, n_init=10, random_state=42)"
3,16,0.445424,12.791209,91,1164,0.854995,"KMeans(n_clusters=16, n_init=10, random_state=42)"
14,27,0.452875,23.969697,33,791,0.911264,"KMeans(n_clusters=27, n_init=10, random_state=42)"


In [16]:
kmeans_labels = best_model.fit_predict(X_scaled)
clustered["kmeans_labels"] = kmeans_labels

In [46]:
# Map each product ID to its KMeans label, and invert that mapping.


clustered["kmeans_labels"] = kmeans_labels

id_to_cluster = dict(zip(clustered['id'].astype('int32'), clustered['kmeans_labels'].astype(np.dtype('int64'))))
cluster_to_ids = clustered.groupby("kmeans_labels")["id"].apply(list).to_dict()

import pickle
with open("id_to_cluster.pkl", "wb") as f:
    pickle.dump(id_to_cluster, f)

with open("cluster_to_ids.pkl", "wb") as f:
    pickle.dump(cluster_to_ids, f)

# # Load the dictionaries back
# with open("id_to_cluster.pkl", "rb") as f:
#     id_to_cluster_loaded = pickle.load(f)

# with open("cluster_to_ids.pkl", "rb") as f:
#     cluster_to_ids_loaded = pickle.load(f)


In [36]:
# Plot
best_model = results_df.loc[results_df['k'] == 24]['model'].iloc[0]
kmeans_labels = best_model.predict(X_scaled)
clustered["kmeans_labels"] = kmeans_labels

import plotly.express as px
fig = px.histogram(
    clustered,
    x="kmeans_labels",
    title="Products per KMeans cluster",
    labels={"kmeans_labels": "Cluster label", "count": "Product count"},
    # xbins={"start": -0.5, "end": 19.5, "size": 1},
)
fig.update_xaxes(dtick=1)
fig.show()



In [57]:
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
    ('quantile', QuantileTransformer(output_distribution='normal')),
    ('scaler', RobustScaler()),
    ('kmeans', best_model)
])
X_raw = df_product_clustering[feature_cols].copy() 
pipeline.fit(X_raw)

model_bundle = {
    'pipeline': pipeline,
    'feature_cols': feature_cols,
    'transformations': ['quantile_normal', 'robust_scaler'],
    'k': best_model.n_clusters,
    'n_samples_trained_on': X_scaled.shape[0],
    'date_trained': pd.Timestamp.now().isocalendar(),
    'Database': 'heynepa_development',
    'Query': query,
    'id_to_cluster_mapping': id_to_cluster,
    'cluster_to_id_mapping': cluster_to_ids,
}


In [58]:
with open("model_bundle.pkl", "wb") as f:
    pickle.dump(model_bundle, f)

In [37]:
silhouette_score(X_scaled, kmeans_labels)

0.2923593102051276

In [43]:
print(clustered["kmeans_labels"].value_counts().sort_index())

kmeans_labels
0     791
1     489
2     636
3     488
4     275
5     233
6     399
7     547
8     358
9     572
10    108
11    577
12    348
13    337
14    494
15    344
16    420
17    451
18    353
19     85
20    443
21    558
22    469
23    266
Name: count, dtype: int64


In [60]:
# load model bundle
with open("model_bundle.pkl", "rb") as f:
    model_bundle = pickle.load(f)

model_bundle.keys()

imported_pipeline = model_bundle['pipeline']

imported_kmeans_label = imported_pipeline.predict(X_raw)

if imported_kmeans_label.all() == kmeans_labels.all():
    print('pass')

pass
